# PDF to Txt Pipeline

### Convert PDF to Text Including Folder Structure

In [3]:
import torch
import cv2
print(cv2.__version__)

print(torch.version.cuda)        # CUDA version PyTorch was built with
print(torch.cuda.get_device_name(0))  # your GPU name

import easyocr
reader = easyocr.Reader(["es"], gpu=True)
print("EasyOCR loaded successfully")

4.13.0
11.8
NVIDIA GeForce RTX 4050 Laptop GPU
EasyOCR loaded successfully


In [ ]:
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from pathlib import Path
from tqdm.auto import tqdm
import json
import os
import easyocr
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH = r"C:\Program Files\Poppler\Library\bin"

pdf_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024")
out_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output")
progress_dir = Path(r"C:\Thesis\data\EIA_data\output_spanish")
out_dir.mkdir(exist_ok=True)
progress_dir.mkdir(exist_ok=True)

MIN_CHARS = 50
DPI = 200
OCR_DPI = 150  # separate from your general DPI

reader = easyocr.Reader(["es"], gpu=True)  # "es" for Spanish

def extract_page_text(pdf_path, page_index, plumber_page):
    text = plumber_page.extract_text() or ""
    if len(text.strip()) >= MIN_CHARS:
        return text, "text"
    images = convert_from_path(
        pdf_path, dpi=OCR_DPI,
        first_page=page_index + 1,
        last_page=page_index + 1,
        poppler_path=POPPLER_PATH
    )
    img_array = np.array(images[0])
    result = reader.readtext(img_array, detail=0, paragraph=True, batch_size=8)
    return "\n".join(result), "OCR-GPU"


_position_lock = threading.Lock()
_positions = set()

def get_position():
    with _position_lock:
        pos = next(i for i in range(3, 20) if i not in _positions)
        _positions.add(pos)
        return pos

def release_position(pos):
    with _position_lock:
        _positions.discard(pos)

import io

def process_pdf(pdf_path):
    relative = pdf_path.relative_to(pdf_dir)
    out_file = (out_dir / relative).with_suffix(".txt")
    progress_file = (progress_dir / relative).with_suffix(".json")

    out_file.parent.mkdir(parents=True, exist_ok=True)
    progress_file.parent.mkdir(parents=True, exist_ok=True)

    if out_file.exists():
        return

    if progress_file.exists():
        try:
            with open(progress_file, encoding='utf-8') as f:
                progress = json.load(f)
            pages_text = progress["pages_text"]
            start_page = progress["next_page"]
        except (json.JSONDecodeError, KeyError):
            progress_file.unlink()
            pages_text = []
            start_page = 0
    else:
        pages_text = []
        start_page = 0

    pos = get_position()
    try:
        # Read PDF as bytes, wrap in BytesIO for pdfplumber
        with open(pdf_path, 'rb') as f:
            pdf_bytes = f.read()
        
        pdf_stream = io.BytesIO(pdf_bytes)
        
        with pdfplumber.open(pdf_stream) as pdf:
            total = len(pdf.pages)
            with tqdm(range(start_page, total), desc=relative.name[:40], position=pos,
                      leave=False, initial=start_page, total=total) as pbar:
                for i in pbar:
                    page = pdf.pages[i]
                    page_text, method = extract_page_text(pdf_bytes, i, page)
                    separator = "=" * 80
                    page_marker = "\n" + separator + "\nPAGE " + str(i+1) + "\n" + separator + "\n"
                    pages_text.append(page_marker + page_text)
                    pbar.set_postfix(method=method)
                    
                    if i % 5 == 0:
                        temp_file = progress_file.with_suffix(".json.tmp")
                        with open(temp_file, "w", encoding='utf-8') as f:
                            json.dump({"pages_text": pages_text, "next_page": i + 1}, f)
                        os.replace(temp_file, progress_file)

        out_file.write_text("".join(pages_text), encoding="utf-8")
        progress_file.unlink(missing_ok=True)
        return

    except Exception as e:
        return f"✗ Failed: {relative}: {str(e)}"

    finally:
        release_position(pos)


def extract_page_text(pdf_bytes, page_index, plumber_page):
    text = plumber_page.extract_text() or ""
    if len(text.strip()) >= MIN_CHARS:
        return text, "text"
    
    # For OCR, write bytes to temp file (pdf2image requires path)
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp:
        tmp.write(pdf_bytes)
        tmp_path = tmp.name
    
    try:
        images = convert_from_path(
            tmp_path, dpi=OCR_DPI,
            first_page=page_index + 1,
            last_page=page_index + 1,
            poppler_path=POPPLER_PATH
        )
        img_array = np.array(images[0])
        result = reader.readtext(img_array, detail=0, paragraph=True, batch_size=8)
        return "\n".join(result), "OCR-GPU"
    finally:
        os.unlink(tmp_path)

if __name__ == "__main__":
    all_pdfs = list(pdf_dir.rglob("*.pdf"))
    WORKERS = 6

    outer = tqdm(total=len(all_pdfs), desc="Total PDFs", position=0)
    
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_pdf, p): p for p in all_pdfs}
    completed = 0
    for future in as_completed(futures):
        result = future.result()
        if result:
            tqdm.write(result)
        completed += 1
        outer.update(1)
        outer.set_description(f"Total PDFs ({completed}/{len(all_pdfs)})")
    
    outer.close()

### Translate Spanish Text to English

In [ ]:
from pathlib import Path
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm
import json
import threading

txt_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output\output_spanish")
out_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output\output_english")
progress_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output\progress_translation")
out_dir.mkdir(exist_ok=True)
progress_dir.mkdir(exist_ok=True)

_position_lock = threading.Lock()
_positions = set()

def get_position():
    with _position_lock:
        pos = next(i for i in range(1, 20) if i not in _positions)
        _positions.add(pos)
        return pos

def release_position(pos):
    with _position_lock:
        _positions.discard(pos)

def translate_text(text):
    chunk_size = 4500
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

all_files = list(txt_dir.rglob("*.txt"))
outer = tqdm(total=len(all_files), desc="Total Files", position=0)

for txt_path in all_files:
    relative = txt_path.relative_to(txt_dir)
    out_file = (out_dir / relative).with_suffix(".txt")
    progress_file = (progress_dir / relative).with_suffix(".json")

    out_file.parent.mkdir(parents=True, exist_ok=True)
    progress_file.parent.mkdir(parents=True, exist_ok=True)

    if out_file.exists():
        outer.update(1)
        continue

    try:
        text = txt_path.read_text(encoding="utf-8")
        chunks = translate_text(text)

        if progress_file.exists():
            with open(progress_file) as f:
                progress = json.load(f)
            translated_chunks = progress["translated_chunks"]
            start_chunk = progress["next_chunk"]
        else:
            translated_chunks = []
            start_chunk = 0

        translator = GoogleTranslator(source="es", target="en")
        pos = get_position()
        BATCH_SIZE = 10

        try:
            with tqdm(range(start_chunk, len(chunks), BATCH_SIZE), desc=relative.name[:40],
                      position=pos, leave=False, total=len(range(start_chunk, len(chunks), BATCH_SIZE))) as pbar:
                for batch_start in pbar:
                    batch = chunks[batch_start:batch_start + BATCH_SIZE]
                    translated_batch = translator.translate_batch(batch)
                    translated_chunks.extend(translated_batch)
                    with open(progress_file, "w") as f:
                        json.dump({"translated_chunks": translated_chunks, "next_chunk": batch_start + len(batch)}, f)
        finally:
            release_position(pos)

        out_file.write_text("\n".join(translated_chunks), encoding="utf-8")
        progress_file.unlink()
        # tqdm.write(f"✓ Done: {relative}")

    except Exception as e:
        tqdm.write(f"✗ Failed: {relative}: {e}")

    outer.update(1)

outer.close()
tqdm.write("All done!")

### Translate Folder Names

In [ ]:
from deep_translator import GoogleTranslator
from pathlib import Path
from tqdm import tqdm
import time

base_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output\output_english")

folders = sorted(
    [p for p in base_dir.rglob("*") if p.is_dir()],
    key=lambda p: len(p.parts),
    reverse=True
)

names = [f.name for f in folders]

CHUNK_SIZE = 50
translated_names = []

for i in tqdm(range(0, len(names), CHUNK_SIZE), desc="Translating"):
    chunk = names[i:i + CHUNK_SIZE]
    translated_names.extend(GoogleTranslator(source="es", target="en").translate_batch(chunk))
    time.sleep(1)

for folder, original_name, translated_name in tqdm(
    zip(folders, names, translated_names), total=len(folders), desc="Renaming"
):
    try:
        if translated_name is None or translated_name == original_name:
            tqdm.write(f"⏭ Unchanged: {original_name}")
            continue

        for char in r'<>:"/\|?*':
            translated_name = translated_name.replace(char, "")

        new_path = folder.parent / translated_name
        folder.rename(new_path)
        tqdm.write(f"✓ {original_name} → {translated_name}")

    except Exception as e:
        tqdm.write(f"✗ Failed: {original_name}: {e}")

print("All done!")

Renaming:  11%|█▏        | 18/160 [00:00<00:00, 175.42it/s]

⏭ Unchanged: Appendix 3B-22.1 Digital Files-Microrouting
⏭ Unchanged: Appendix 3B-39.1 Digital Files
⏭ Unchanged: IDF ELEVATIONS
⏭ Unchanged: PROCESSING PLANT ELEVATIONS
✓ Anexo 3A-03 Fichas de Medición de receptores de flujo vehicular → Annex 3A-03 Vehicular flow receiver measurement sheets
✓ Anexo 3A-04 Fichas de Medición de Ruido en Fauna → Annex 3A-04 Noise Measurement Sheets in Wildlife
✓ Anexo 3A-05 Certificados de Calibración → Annex 3A-05 Calibration Certificates
✓ Anexo 3A-06 Archivos Digitales de Ruido (kmz y shp) → Annex 3A-06 Digital Noise Files (kmz and shp)
✓ Anexo 3A-07 Línea de Base de Ruido y Vibraciones → Annex 3A-07 Noise and Vibration Baseline
✓ Anexo 3A-08 Archivos digitales de geología → Annex 3A-08 Geology Digital Files
✓ Anexo 3A-09 Archivos digitales de Geomorfología → Annex 3A-09 Geomorphology digital files
✓ Anexo 3A-10 Archivos digitales de Riesgos geológicos y geomorfológicos → Annex 3A-10 Digital files of geological and geomorphological risks
✓ Anexo 3A-11

Renaming:  36%|███▌      | 57/160 [00:00<00:00, 162.09it/s]

✓ Anexo 3B-29 Distribución Espacial de Hongos → Annex 3B-29 Spatial Distribution of Fungi
✓ Anexo 3B-30 Distribución Espacial de Líquenes → Annex 3B-30 Spatial Distribution of Lichens
✓ Anexo 3B-31 Planillas Darwin Core Componente Hongos y Líquenes → Annex 3B-31 Darwin Core Worksheets Component Fungi and Lichens
✓ Anexo 3B-33 Archivos digitales de fauna (kmz y shp) → Annex 3B-33 Digital fauna files (kmz and shp)
✓ Anexo 3B-39 Estudio de Monitoreo de Monito del Monte → Annex 3B-39 Monito del Monte Monitoring Study
✓ Anexo 3B-40 Planilla Darwin Core Componente Fauna Silvestre → Annex 3B-40 Darwin Core Worksheet Component Wildlife
✓ Anexo 3B-42 Archivos Digitales de Fauna Invertebrada (kmz y shp) → Annex 3B-42 Digital Files of Invertebrate Fauna (kmz and shp)
✓ Anexo 3B-43 Planilla Darwin Core Componente Fauna Invertebrada → Annex 3B-43 Darwin Core Worksheet Component Invertebrate Fauna
✓ Anexo 3B-46 Archivos digitales de Ecosistemas Acuáticos Continentales (kmz y shp) → Annex 3B-46 Digit

Renaming:  48%|████▊     | 76/160 [00:00<00:00, 168.41it/s]

...

Renaming:  70%|███████   | 112/160 [00:00<00:00, 161.97it/s]

✓ Anexos Sección B → Annexes Section B
✓ Anexo 3C-50 Fichas de registro hallazgos arqueológicos → Annex 3C-50 Archaeological findings record sheets
✓ Anexo 3C-51 Línea de Base Arqueología → Annex 3C-51 Baseline Archeology
✓ Anexo 3C-52 Informe Sondeos Arqueológicos → Annex 3C-52 Archaeological Survey Report
✓ Anexo 3C-53 Archivos digitales de Paleontología (kmz y shp) → Annex 3C-53 Paleontology digital files (kmz and shp)
✓ Anexo 3C-54 Características Cuencas Visuales → Annex 3C-54 Viewshed Characteristics
✓ Anexo 3C-55 Archivos digitales de Paisaje (kmz y shp) → Annex 3C-55 Digital Landscape Files (kmz and shp)
✓ Anexo 3C-56 Archivos digitales Áreas Protegidas y Sitios Prioritarios (kmz y shp) → Annex 3C-56 Digital files Protected Areas and Priority Sites (kmz and shp)
✓ Anexo 3C-57 Archivos digitales de Turismo (kmz y shp) → Annex 3C-57 Tourism digital files (kmz and shp)
✓ Anexo 3C-58 Archivos digitales Uso de Territorio (kmz y shp) → Annex 3C-58 Digital files Territory Use (kmz and

✓ Anexo 7-2 Modelación de medidas de compensación → Annex 7-2 Modeling of compensation measures
✓ Anexo 7-3 Caracterización del Sitio → Annex 7-3 Site Characterization
✓ Apéndice 10-11.1 Planos → Appendix 10-11.1 Plans
✓ Apéndice 10-11.2 Archivos digitales → Appendix 10-11.2 Digital Files
✓ Anexos → Annexes
✓ Anexos → Annexes
✓ Anexos → Annexes
✓ Anexo 1-2 Cartografía digital del Proyecto → Annex 1-2 Digital cartography of the Project
✓ Anexo 1-3 Estimación de emisiones atmosféricas → Annex 1-3 Estimation of atmospheric emissions
✓ Anexo 1-9 HDS reactivos del proceso → Annex 1-9 process reagent SDS
✓ Anexos_Sección_A → Annexes_Section_A
✓ Anexos_Sección_B → Annexes_Section_B
✓ Anexos_Sección_C → Annexes_Section_C
✓ Anexos_Sección_D → Annexes_Section_D
✓ Anexo_10-13_PASM_157 → Annex_10-13_PASM_157
✓ Anexo_10-14_PASM_160 → Annex_10-14_PASM_160
✓ Anexo_10-3_PASM_136 → Annex_10-3_PASM_136
✓ Anexo_10-4._PASM_137 → Annex_10-4._PASM_137
✓ Anexo_10-5_PASM_138 → Annex_10-5_PASM_138
✓ Anexo_10-6

Renaming: 100%|██████████| 160/160 [00:00<00:00, 168.17it/s]

✓ Capítulo_15_Acciones_Previas → Chapter_15_Previous_Actions
✓ Capítulo_15_Acciones_Previas_77 → Chapter_15_Previous_Actions_77
✓ Participación_Ciudadana_Temprana → Early_Citizen_Participation
All done!


### Translate File Names

In [ ]:
from deep_translator import GoogleTranslator
from pathlib import Path
import time
from tqdm import tqdm

english_dir = Path(r"C:\Thesis\data\EIA_data\EIA_2024_output\output_english")

CHUNK_SIZE = 50

def clean_filename(name):
    for char in r'<>:"/\|?*':
        name = name.replace(char, "")
    return name.strip()

files = list(english_dir.rglob("*.txt"))
stems = [f.stem for f in files]

translated_stems = []
for i in tqdm(range(0, len(stems), CHUNK_SIZE), desc="Translating"):
    chunk = stems[i:i + CHUNK_SIZE]
    results = GoogleTranslator(source="es", target="en").translate_batch(chunk)
    translated_stems.extend([clean_filename(r) for r in results])
    time.sleep(1)

for file, original_stem, translated_stem in tqdm(
    zip(files, stems, translated_stems), total=len(files), desc="Renaming"
):
    try:
        if translated_stem is None or translated_stem == original_stem:
            tqdm.write(f"⏭ Unchanged: {original_stem}")
            continue

        new_path = file.parent / (translated_stem + file.suffix)
        file.rename(new_path)
        tqdm.write(f"✓ {original_stem} → {translated_stem}")

    except Exception as e:
        tqdm.write(f"✗ Failed: {original_stem}: {e}")

print("\nAll done!")

Renaming:   6%|▋         | 19/295 [00:00<00:01, 187.24it/s]

✓ Anexo_1-10_Análisis_de_escenarios_climáticos_del_proyecto_CCG_UC → Annex_1-10_Analysis_of_climatic_scenarios_of_the_CCG_UC_project
✓ Anexo_1-11_Evaluación_de_riesgos_climáticos → Annex_1-11_Climate_risk_assessment
✓ Anexo_1-1_Antecedentes_legales → Annex_1-1_Legal_Background
✓ Anexo_1-1_Antecedentes_legales_82 → Annex_1-1_Legal_Background_82
✓ Anexo_1-4_Plan_de_aplicación_de_supresor_de_polvo → Annex 1-4 Dust Suppressant Application Plan
✓ Anexo_1-5_Plan_de_Mantencion_de_Caminos → Annex_1-5_Road_Maintenance_Plan
✓ Anexo_1-6_HDS_producto_caminos_no_pavimentados → Annex_1-6_HDS_product_unpaved_roads
✓ Anexo_1-7_Plan_de_Compensación_de_emisiones → Annex_1-7_Emissions_Compensation_Plan
✓ Anexo_1-8_Caracterización_Geoquímica_Arcillas_por_Procesar → Annex_1-8_Characterization_Geochemistry_Clays_to_Process
✓ Anexo_10-10_PASM_146_Invertebrados → Annex_10-10_PASM_146_Invertebrates
✓ Anexo_10-11_PASM_149 → Annex_10-11_PASM_149
✓ Anexo_10-12_PAS_156 → Annex_10-12_PAS_156
✓ Anexo_10-1_PAS_119 → 

✓ Capítulo_10_Normativa_Ambiental_Aplicable → Chapter_10_Applicable_Environmental_Regulations
✓ Capítulo_11_Relación_con_planes_de_desarrollo → Chapter_11_Relationship_with_development_plans
✓ Capítulo_12_Relación_con_PEE → Chapter_12_Relationship_with_PEE
✓ Capítulo_13_Compromisos_Ambientales_Voluntarios → Chapter_13_Voluntary_Environmental_Commitments
✓ Capítulo_14._Fichas_Resumen → Chapter_14._Summary_Files
✓ Capítulo_17_Listado_de_profesionales → Chapter_17_List_of_professionals
✓ Desarrollo_del_proyecto_por_etapas → Project_development_by_stages
✓ Descripción_cronológica_de_las_fases_del_proyecto → Chronological_description_of_the_project_phases
✓ Establecimiento_inicio_ejecución_del_proyecto → Establishment_start_project_execution
✓ Extracto → Extract
✓ Monitoreo_participativo → Participatory_monitoring
✓ Índice_general_EIA → EIA_general_index
✓ Anexo 1-2 Cartografía digital del Proyecto → Annex 1-2 Digital cartography of the Project
✓ Anexo 1-2.2 A2 - Plano General de Instalacio

Renaming:  36%|███▌      | 105/295 [00:00<00:00, 207.90it/s]

✓ Anexo 3A-05 Certificados de Calibración → Annex 3A-05 Calibration Certificates
✓ Anexo 3A-06 Portada Archivos digitales de Ruido → Annex 3A-06 Cover Digital Noise Files
✓ Anexo 3A-07 Línea de Base de Ruido y Vibraciones_Rev.0 → Annex 3A-07 Noise and Vibration Baseline_Rev.0
✓ Anexo 3A-08 Portada Archivos Digitales de Geología → Annex 3A-08 Cover Geology Digital Files
✓ Anexo 3A-09 Portada Archivos Digitales de Geomorfología → Annex 3A-09 Cover Geomorphology Digital Files
✓ Anexo 3A-10 Portada Archivos Digitales de Riesgos Geológicos y Geomorfológicos → Annex 3A-10 Cover of Digital Files of Geological and Geomorphological Risks
✓ Anexo 3A-11 Portada Archivos Digitales de Hidrología → Annex 3A-11 Cover Hydrology Digital Files
✓ Anexo 3A-12 Línea Base Hidrológica → Annex 3A-12 Hydrological Baseline
✓ Anexo 3A-13 Portada Archivos Digitales de Hidrogeología → Annex 3A-13 Cover Hydrogeology Digital Files
✓ Anexo 3A-14 Línea de Base Hidrogeológica_Rev.0 → Annex 3A-14 Hydrogeological Baselin

...

✓ Anexo 4-1 Modelación meteorológica y de calidad del aire_Rev.0 → Annex 4-1 Meteorological and air quality modeling_Rev.0
✓ Apéndice 4-1.1 Archivos de modelación → Appendix 4-1.1 Modeling files
✓ Apéndice 4-1.2 Documento de referencia → Appendix 4-1.2 Reference document
✓ Anexo 4-3 Estudio de Impacto Vial EIA Rev0 → Annex 4-3 Road Impact Study EIA Rev0
✓ Apéndice 4-3.1 Mediciones de Flujo Vehicular_Rev.0 → Appendix 4-3.1 Vehicular Flow Measurements_Rev.0
✓ Apéndice 4-3.2 Respuestas a Consultas por Ley de Transparencia_Rev.0 → Appendix 4-3.2 Responses to Queries by Transparency Law_Rev.0
✓ Apéndice 4-3.3 Archivos de modelación vial → Appendix 4-3.3 Road modeling files
✓ Anexo 4-4. Mapa de Riesgo de Erosión_Rev.0 → Annex 4-4. Erosion Risk Map_Rev.0
✓ Apéndice 4-4.1 Archivos digitales → Appendix 4-4.1 Digital files
✓ Anexo 4-5. Matriz Causa-Efecto → Annex 4-5. Cause-Effect Matrix
✓ Anexo 5-1 Estudio de Riesgo a la Salud de Población Rev.0 Final → Annex 5-1 Population Health Risk Study Re

Renaming:  91%|█████████ | 267/295 [00:01<00:00, 158.25it/s]

✓ A2 - PAS 149 Plano de Intervencion 2 de 12 → A2 - PAS 149 Intervention Plan 2 of 12
✓ A2 - PAS 149 Plano de Intervencion 3 de 12 → A2 - PAS 149 Intervention Plan 3 of 12
✓ A2 - PAS 149 Plano de Intervencion 4 de 12 → A2 - PAS 149 Intervention Plan 4 of 12
✓ A2 - PAS 149 Plano de Intervencion 5 de 12 → A2 - PAS 149 Intervention Plan 5 of 12
✓ A2 - PAS 149 Plano de Intervencion 6 de 12 → A2 - PAS 149 Intervention Plan 6 of 12
✓ A2 - PAS 149 Plano de Intervencion 7 de 12 → A2 - PAS 149 Intervention Plan 7 of 12
✓ A2 - PAS 149 Plano de Intervencion 8 de 12 → A2 - PAS 149 Intervention Plan 8 of 12
✓ A2 - PAS 149 Plano de Intervencion 9 de 12 → A2 - PAS 149 Intervention Plan 9 of 12
✓ A2 - PAS 149 Plano General → A2 - PAS 149 General Plan
✓ Apéndice 10-11.1 Planos → Appendix 10-11.1 Plans
✓ Apéndice 10-11.2 Archivos digitales → Appendix 10-11.2 Digital Files
✓ Capitulo 15 Acciones Previas_Rev.0 → Chapter 15 Previous Actions_Rev.0
✓ Anexo 15-1 Identificación Organizaciones Sociales_Rev.0 → 

Renaming: 100%|██████████| 295/295 [00:01<00:00, 177.67it/s]

✓ Anexo 15-2 Asamblea comunitaria_Rev.0 → Annex 15-2 Community Assembly_Rev.0
✓ Anexo 15-3 Comentarios totales_Rev.0 → Annex 15-3 Total Comments_Rev.0
✓ Anexo 15-4 Lista asistencia reuniones con ciudadanía_Rev.0 → Annex 15-4 List of attendance at meetings with citizens_Rev.0
✓ Anexo 15-5 Minutas de reuinión_Rev.0 → Annex 15-5 Meeting Minutes_Rev.0
✓ Anexo 15-6 Reuniones Lobby_Rev.0 → Annex 15-6 Lobby Meetings_Rev.0
✓ Anexo 15-7 Encuesta Penco Ekhos_Rev.0 → Annex 15-7 Penco Ekhos Survey_Rev.0
✓ Anexo 15-8 Estudio de opinión pública digital_Rev.0 → Annex 15-8 Digital public opinion study_Rev.0
✓ Anexo 15-9 Informe Ejecutivo Diagnóstico Comunal Penco_Rev.0 → Annex 15-9 Penco Communal Diagnosis Executive Report_Rev.0
✓ Capitulo 15 Acciones Previas_Rev.0 → Chapter 15 Previous Actions_Rev.0
✓ Anexo 15-1 Identificación Organizaciones Sociales_Rev.0 → Annex 15-1 Identification of Social Organizations_Rev.0
✓ Anexo 15-2 Asamblea comunitaria_Rev.0 → Annex 15-2 Community Assembly_Rev.0
✓ Anexo 15